Pipeline Version 3 для IEEE Fraud Detection

Цели:
0. Заменить регрессионные модели на классификационные, получить корректные метрики и заполнить таблицу результатов.
1. На выходе пайплайна получить и сохранить два файла: X.csv (таблица признаков) и y.csv (вектор таргета).
2. Реализовать кастомный PyTorch Dataset, возвращающий кортеж (features, label).
3. Определить и обучить простую нейронную сеть на PyTorch для бинарной классификации.
4. Провести серию экспериментов, целью которых — превзойти F1 классических моделей, и отразить результаты в таблице.


Импорты и настройки

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Параметры
SAMPLE_FRAC = 0.3       # доля данных для подвыборки
NAN_THRESH = 0.6        # порог для удаления колонок с пропусками
CAT_ONEHOT_THRESH = 10  # <= unique -> OHE, иначе frequency encoding
SEED = 42
BATCH_SIZE = 256
EPOCHS = 10
LR = 1e-3


Функция загрузки данных с рандомной подвыборкой

In [2]:
def load_data(sample_frac=SAMPLE_FRAC, seed=SEED):
    train_trans = pd.read_csv('drive-download-20250512T040309Z-1-001/train_transaction.csv')
    train_id    = pd.read_csv('drive-download-20250512T040309Z-1-001/train_identity.csv')
    test_trans  = pd.read_csv('drive-download-20250512T040309Z-1-001/test_transaction.csv')
    test_id     = pd.read_csv('drive-download-20250512T040309Z-1-001/test_identity.csv')
    train = pd.merge(train_trans, train_id, on='TransactionID', how='left')
    test  = pd.merge(test_trans, test_id, on='TransactionID', how='left')
    train = train.sample(frac=sample_frac, random_state=seed).reset_index(drop=True)
    return train, test

EDA

In [3]:
def eda_report(df):
    print('Shape:', df.shape)
    print('Fraud ratio:', df['isFraud'].mean())
    print('\nTop NaN columns:')
    print(df.isna().mean().sort_values(ascending=False).head(10))
    cat_cols = df.select_dtypes(include=['object']).columns
    print('\nCategorical unique counts:')
    print(df[cat_cols].nunique().sort_values().head(10))

Предобработка и сохранение X, y

In [4]:
def preprocess_and_save(df_train, df_test):
    # Целевая переменная
    y = df_train['isFraud']
    # Удалим идентификаторы
    df_train = df_train.drop(['isFraud','TransactionID'], axis=1)
    df_test  = df_test.drop(['TransactionID'], axis=1)
    # Одинаковые колонки
    common = list(set(df_train.columns)&set(df_test.columns))
    df_train = df_train[common]
    df_test  = df_test[common]
    # Удаление по NaN
    nan_frac = df_train.isna().mean()
    drop_cols = nan_frac[nan_frac>NAN_THRESH].index.tolist()
    df_train.drop(drop_cols, axis=1, inplace=True)
    df_test.drop(drop_cols, axis=1, inplace=True)
    # Имена колонок
    cat_cols = df_train.select_dtypes(include=['object']).columns
    num_cols = df_train.select_dtypes(include=['int64','float64']).columns
    X_train_enc = pd.DataFrame(); X_test_enc = pd.DataFrame()
    # Кодирование категорий
    for c in cat_cols:
        n_uniq = df_train[c].nunique()
        if n_uniq<=CAT_ONEHOT_THRESH:
            ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            arr_tr = ohe.fit_transform(df_train[[c]])
            arr_te = ohe.transform(df_test[[c]])
            cols = [f"{c}_{v}" for v in ohe.categories_[0]]
            X_train_enc[cols] = arr_tr; X_test_enc[cols] = arr_te
        else:
            freq = df_train[c].value_counts(normalize=True)
            X_train_enc[c+'_freq'] = df_train[c].map(freq)
            X_test_enc[c+'_freq']  = df_test[c].map(freq).fillna(0)
    # Масштабирование числовых
    scaler = StandardScaler()
    X_train_enc[num_cols] = scaler.fit_transform(df_train[num_cols])
    X_test_enc[num_cols]  = scaler.transform(df_test[num_cols])
    # Уберём NaN
    X_train_enc.fillna(0, inplace=True); X_test_enc.fillna(0, inplace=True)
    # Сохраняем
    X_train_enc.to_csv('X.csv', index=False)
    y.to_csv('y.csv', index=False)
    print('Сохранены X.csv и y.csv')
    return X_train_enc, X_test_enc, y

Классические классификаторы и метрики

In [5]:
def train_classical(X, y):
    X_tr, X_val, y_tr, y_val = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y)
    results = []
    # Decision Tree
    dt = DecisionTreeClassifier(random_state=SEED)
    dt.fit(X_tr, y_tr)
    y_pred = dt.predict(X_val)
    results.append(('DecisionTree', f1_score(y_val,y_pred)))
    # Bagging
    bg = BaggingClassifier(random_state=SEED)
    bg.fit(X_tr, y_tr)
    y_pred = bg.predict(X_val)
    results.append(('Bagging', f1_score(y_val,y_pred)))
    # XGBoost
    xgb_cl = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED)
    xgb_cl.fit(X_tr, y_tr)
    y_pred = xgb_cl.predict(X_val)
    results.append(('XGBoost', f1_score(y_val,y_pred)))
    return results



 PyTorch Dataset

In [6]:
class FraudDataset(Dataset):
    def __init__(self, X: pd.DataFrame, y: pd.Series):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

Простая нейронная сеть PyTorch

In [7]:
class FraudNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

Тренировочный цикл для PyTorch

In [8]:
def train_nn(X, y):
    ds = FraudDataset(X, y)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)
    model = FraudNet(X.shape[1])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    for epoch in range(EPOCHS):
        model.train()
        for xb, yb in dl:
            pred = model(xb).squeeze()
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    # Оценка
    model.eval()
    with torch.no_grad():
        preds = model(ds.X).squeeze().numpy()>=0.5
        f1 = f1_score(ds.y.numpy(), preds)
    return f1

Эксперименты и таблица результатов

In [9]:

train, test = load_data()
eda_report(train)
X, X_test, y = preprocess_and_save(train, test)
classical = train_classical(X,y)
f1_nn = train_nn(X, y)
# Таблица результатов
results = classical + [('NeuralNet', f1_nn)]
df_res = pd.DataFrame(results, columns=['Model','F1'])
display(df_res)



Shape: (177162, 434)
Fraud ratio: 0.03579209988598006

Top NaN columns:
id_24    0.992182
id_25    0.991482
id_21    0.991471
id_08    0.991465
id_07    0.991465
id_26    0.991443
id_27    0.991437
id_23    0.991437
id_22    0.991437
dist2    0.936843
dtype: float64

Categorical unique counts:
M2       2
M3       2
M1       2
id_12    2
M8       2
M7       2
M6       2
M5       2
M9       2
id_36    2
dtype: int64
Сохранены X.csv и y.csv


,Model,F1
0,DecisionTree,0.475038
1,Bagging,0.574005
2,XGBoost,0.599793
3,NeuralNet,0.622133
